<a href="https://colab.research.google.com/github/javageek2018/AirlineArrivalDelay/blob/%E2%80%9Cflight_data%E2%80%9D/get_data_bts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from zipfile import ZipFile
from io import BytesIO
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_bts_month(year, month, usecols=None):
    url = (
        f"https://transtats.bts.gov/PREZIP/"
        f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    z = ZipFile(BytesIO(response.content))
    csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
    df = pd.read_csv(z.open(csv_name), low_memory=False, usecols=usecols)
    df["Year"] = year
    df["Month"] = month
    return df

columns = [
    "FlightDate", "Reporting_Airline",
    "Origin", "OriginCityName", "Dest", "DestCityName",
    "DepDelay", "DepDelayMinutes",
    "ArrDelay", "ArrDelayMinutes",
    "Cancelled", "Diverted",
    "CarrierDelay", "WeatherDelay",
    "NASDelay", "SecurityDelay", "LateAircraftDelay",
    "AirTime", "Distance",
]

# Build list of (year, month) pairs
tasks = [(y, m) for y in range(2018, 2025) for m in range(1, 13)]

frames = []
failed = []

# Download 4 months in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(load_bts_month, y, m, columns): (y, m)
        for y, m in tasks
    }
    for future in as_completed(futures):
        y, m = futures[future]
        try:
            chunk = future.result()
            frames.append(chunk)
            print(f"✅ {y}-{m:02d}: {chunk.shape[0]:,} rows")
        except Exception as e:
            print(f"⚠️  {y}-{m:02d}: Skipped ({e})")
            failed.append((y, m))

df = pd.concat(frames, ignore_index=True)
df = df.sort_values(["FlightDate"]).reset_index(drop=True)

print(f"\n🎉 Total: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📅 Range: {df['FlightDate'].min()} → {df['FlightDate'].max()}")

# Save immediately
df.to_parquet("flights_2018_2024.parquet", index=False)
print("💾 Saved to flights_2018_2024.parquet")

✅ 2018-02: 520,769 rows
✅ 2018-01: 570,138 rows
✅ 2018-03: 612,034 rows
✅ 2018-04: 596,078 rows
✅ 2018-05: 616,573 rows
✅ 2018-08: 637,103 rows
✅ 2018-06: 626,217 rows
✅ 2018-07: 645,317 rows
✅ 2018-09: 585,765 rows
✅ 2018-11: 586,231 rows
✅ 2018-10: 616,128 rows
✅ 2018-12: 593,842 rows
✅ 2019-01: 583,985 rows
✅ 2019-02: 533,175 rows
✅ 2019-03: 632,074 rows
✅ 2019-05: 636,390 rows
✅ 2019-04: 612,023 rows
✅ 2019-06: 636,691 rows
✅ 2019-07: 659,029 rows
✅ 2019-09: 605,979 rows
✅ 2019-08: 658,461 rows
✅ 2019-10: 636,014 rows
✅ 2019-11: 602,453 rows
✅ 2020-01: 607,346 rows
✅ 2019-12: 625,763 rows
✅ 2020-05: 180,617 rows
✅ 2020-04: 313,382 rows
✅ 2020-03: 648,229 rows
✅ 2020-02: 574,268 rows
✅ 2020-06: 223,732 rows
✅ 2020-07: 352,888 rows
✅ 2020-09: 323,347 rows
✅ 2020-10: 352,106 rows
✅ 2020-08: 376,715 rows
✅ 2020-11: 364,367 rows
✅ 2020-12: 371,357 rows
✅ 2021-02: 332,468 rows
✅ 2021-01: 361,428 rows
✅ 2021-03: 444,476 rows
✅ 2021-05: 495,544 rows
✅ 2021-04: 450,637 rows
✅ 2021-06: 546,1

In [ ]:
import pandas as pd
from zipfile import ZipFile
from io import BytesIO
import requests

year, month = 2024, 1
url = (
    f"https://transtats.bts.gov/PREZIP/"
    f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
)

response = requests.get(url)
with ZipFile(BytesIO(response.content)) as z:
    csv_name = z.namelist()[0]
    df = pd.read_csv(z.open(csv_name), nrows=0)  # headers only

print(f"Total columns: {len(df.columns)}")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

Total columns: 110
  1. Year
  2. Quarter
  3. Month
  4. DayofMonth
  5. DayOfWeek
  6. FlightDate
  7. Reporting_Airline
  8. DOT_ID_Reporting_Airline
  9. IATA_CODE_Reporting_Airline
 10. Tail_Number
 11. Flight_Number_Reporting_Airline
 12. OriginAirportID
 13. OriginAirportSeqID
 14. OriginCityMarketID
 15. Origin
 16. OriginCityName
 17. OriginState
 18. OriginStateFips
 19. OriginStateName
 20. OriginWac
 21. DestAirportID
 22. DestAirportSeqID
 23. DestCityMarketID
 24. Dest
 25. DestCityName
 26. DestState
 27. DestStateFips
 28. DestStateName
 29. DestWac
 30. CRSDepTime
 31. DepTime
 32. DepDelay
 33. DepDelayMinutes
 34. DepDel15
 35. DepartureDelayGroups
 36. DepTimeBlk
 37. TaxiOut
 38. WheelsOff
 39. WheelsOn
 40. TaxiIn
 41. CRSArrTime
 42. ArrTime
 43. ArrDelay
 44. ArrDelayMinutes
 45. ArrDel15
 46. ArrivalDelayGroups
 47. ArrTimeBlk
 48. Cancelled
 49. CancellationCode
 50. Diverted
 51. CRSElapsedTime
 52. ActualElapsedTime
 53. AirTime
 54. Flights
 55. Distance
 5

In [ ]:
import pandas as pd
from zipfile import ZipFile
from io import BytesIO
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_bts_month(year, month, usecols=None):
    url = (
        f"https://transtats.bts.gov/PREZIP/"
        f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    z = ZipFile(BytesIO(response.content))
    csv_name = [f for f in z.namelist() if f.endswith('.csv')][0]
    df = pd.read_csv(z.open(csv_name), low_memory=False, usecols=usecols)
    df["Year"] = year
    df["Month"] = month
    return df

columns = [
"Year","Quarter","Month","DayofMonth","DayOfWeek","FlightDate","Reporting_Airline",
"Flight_Number_Reporting_Airline","Origin","Dest","CRSDepTime","DepTimeBlk",
"CRSArrTime","ArrDel15","CRSElapsedTime","Distance","DistanceGroup"
]

# Build list of (year, month) pairs
tasks = [(y, m) for y in range(2018, 2025) for m in range(1, 13)]

frames = []
failed = []

# Download 4 months in parallel
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {
        executor.submit(load_bts_month, y, m, columns): (y, m)
        for y, m in tasks
    }
    for future in as_completed(futures):
        y, m = futures[future]
        try:
            chunk = future.result()
            frames.append(chunk)
            print(f"✅ {y}-{m:02d}: {chunk.shape[0]:,} rows")
        except Exception as e:
            print(f"⚠️  {y}-{m:02d}: Skipped ({e})")
            failed.append((y, m))

df = pd.concat(frames, ignore_index=True)
df = df.sort_values(["FlightDate"]).reset_index(drop=True)

print(f"\n🎉 Total: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📅 Range: {df['FlightDate'].min()} → {df['FlightDate'].max()}")

# Save immediately
df.to_parquet("flights_2018_2024_v2.parquet", index=False)
print("💾 Saved to flights_2018_2024_v2.parquet")

✅ 2018-02: 520,769 rows
✅ 2018-01: 570,138 rows
✅ 2018-03: 612,034 rows
✅ 2018-04: 596,078 rows
✅ 2018-06: 626,217 rows
✅ 2018-05: 616,573 rows
✅ 2018-07: 645,317 rows
✅ 2018-08: 637,103 rows
✅ 2018-10: 616,128 rows
✅ 2018-09: 585,765 rows
✅ 2018-12: 593,842 rows
✅ 2019-01: 583,985 rows
✅ 2018-11: 586,231 rows
✅ 2019-02: 533,175 rows
✅ 2019-03: 632,074 rows
✅ 2019-04: 612,023 rows
✅ 2019-06: 636,691 rows
✅ 2019-05: 636,390 rows
✅ 2019-08: 658,461 rows
✅ 2019-09: 605,979 rows
✅ 2019-07: 659,029 rows
✅ 2019-10: 636,014 rows
✅ 2019-11: 602,453 rows
✅ 2020-02: 574,268 rows
✅ 2020-03: 648,229 rows
✅ 2020-05: 180,617 rows
✅ 2020-01: 607,346 rows
✅ 2020-04: 313,382 rows
✅ 2020-06: 223,732 rows
✅ 2019-12: 625,763 rows
✅ 2020-08: 376,715 rows
✅ 2020-07: 352,888 rows
✅ 2020-09: 323,347 rows
✅ 2020-10: 352,106 rows
✅ 2020-12: 371,357 rows
✅ 2021-01: 361,428 rows
✅ 2020-11: 364,367 rows
✅ 2021-02: 332,468 rows
✅ 2021-04: 450,637 rows
✅ 2021-05: 495,544 rows
✅ 2021-03: 444,476 rows
✅ 2021-06: 546,1